In [ ]:
!pip install -U langchain-community langchain-text-splitters langchain-groq langchain-huggingface langdetect python-dotenv langchain


import os
from dotenv import load_dotenv
import langchain_community
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langdetect import detect, LangDetectException
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [6]:
# import API from env in project folder

from google.colab import userdata
groq_key=userdata.get('Groq')
hf_key=userdata.get('HuggingFace_API_Key')
print(f"Groq Key (first 5 chars):", groq_key is not None)
print("HF key loaded:", hf_key is not None)

Groq Key (first 5 chars): True
HF key loaded: True


In [ ]:
# load transcript from youtbe
youtube_url="https://www.youtube.com/watch?v=e7hNWADI0H8"
loader= YoutubeLoader.from_youtube_url(youtube_url,add_video_info=False, language=["en","hi","ur"])
transcript=loader.load()


In [ ]:
#splitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
splitted_text=text_splitter.split_documents(transcript)

In [ ]:
# base llm
llm_translation=ChatGroq(groq_api_key=groq_key, model="llama-3.3-70b-versatile", temperature=0, max_tokens=512)


# persona classification
prompt_translation=PromptTemplate(
    input_variables=["text"],
    template="""Translate the following text to English.
              Keep the meaning accurate and concise. Don't add phrase The translation of {text} to English is:.
               text:{text}""")

#translation chain
chain_translation=prompt_translation| llm_translation| StrOutputParser()

In [ ]:
#logic to translate hindi script in english for better embedding
translated_split=[]
for doc in splitted_text:
    text = doc.page_content.strip()

    try:
        lang = detect(text)
    except LangDetectException:
        # If detection fails, assume English
        lang = "en"

    if lang == "hi" or lang== "ur":
        # Translate → English
        response=chain_translation.invoke({"text":text})
        doc.page_content=response
        doc.metadata["translated_from"] = "hi"
    else:
        # Already English → keep as is
        doc.metadata["translated_from"] = "en"

    translated_split.append(doc)


splitted_text=translated_split

In [ ]:
# embedding
embedding=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-l6-v2")


In [ ]:
#vector store
vector_store=FAISS.from_documents(splitted_text,embedding)

In [ ]:
retriever1=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":2})

Augmentation and Generation part of RAG

In [ ]:
main_llm = ChatGroq(groq_api_key=groq_key, model="llama-3.3-70b-versatile", temperature=0.2, max_tokens=512)

In [ ]:
main_prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
def format_retrieved_text(retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
def translate_question(text):
    try:
        lang = detect(text)
    except LangDetectException:
        # If detection fails, assume English
        print("Please write valid Urdu, Hindi or English language.")

    if lang == "hi" or lang=="ur":
        # Translate Hindi → English
        response=chain_translation.invoke({"text":text})
        final_question=response
    elif lang=="en":
        final_question=text
    else:
        print("Please write valid Urdu, Hindi or English language.")
        final_question=""
    return final_question


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever1 | RunnableLambda(format_retrieved_text),
    'question': RunnableLambda(translate_question)
})

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain=parallel_chain | main_prompt | main_llm | parser

In [ ]:
main_chain.invoke("is video is specifically about linkedin ")

'Yes, the video appears to be specifically about LinkedIn, as the speaker is guiding the viewer on how to use the platform, searching for a specific company page ("Deposink"), and navigating its features.'